In [1]:
import pandas as pd
import numpy as np
import spacy
from sentence_transformers import SentenceTransformer, util
from sklearn.metrics import precision_score, recall_score, f1_score

print("Dependencies loaded successfully.")

e:\AI-Resume-Evaluator\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dependencies loaded successfully.


In [2]:
# 1. Load the NLP model for entity extraction (NER)
# Starting with a lightweight spaCy model, which we can upgrade to a custom Transformer later
nlp = spacy.load("en_core_web_sm")

# 2. Load the Embedding model for Semantic Scoring
# all-MiniLM-L6-v2 is fast, highly accurate, and perfect for semantic textual similarity
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print("NLP and Embedding models initialized.")

NLP and Embedding models initialized.


In [3]:
def evaluate_thresholds(jd_skills, candidate_skills, thresholds):
    """
    Computes cosine similarity between JD skills and Candidate skills,
    testing multiple thresholds to find the optimal F1-Score sweet spot.
    """
    # Generate vector embeddings
    jd_embeddings = embedder.encode(jd_skills, convert_to_tensor=True)
    candidate_embeddings = embedder.encode(candidate_skills, convert_to_tensor=True)
    
    # Calculate Cosine Similarity matrix
    cosine_scores = util.cos_sim(jd_embeddings, candidate_embeddings)
    
    results = {}
    
    # Grid Search over thresholds
    for thresh in thresholds:
        # If score > thresh, it's a predicted match (1), else (0)
        # Note: In a real test, you'd compare these predictions against ground-truth manual labels
        matches = (cosine_scores > thresh).int()
        
        # Storing raw matches for the boilerplate. 
        # To calculate exact F1, you will plug in your y_true array here.
        results[thresh] = matches.numpy()
        
    return results

# Sample Data for Prototyping
job_requirements = ["Machine Learning", "React.js", "Python Backend"]
parsed_resume_skills = ["ML", "React", "NodeJS", "Python"]
test_thresholds = [0.70, 0.75, 0.80, 0.85, 0.90]

# Run the grid search
grid_results = evaluate_thresholds(job_requirements, parsed_resume_skills, test_thresholds)

print("Grid Search complete. Ready for F1-Score evaluation mapping.")

Grid Search complete. Ready for F1-Score evaluation mapping.


In [4]:
# Cell 4: Custom Entity Extraction with spaCy's EntityRuler
from spacy.pipeline import EntityRuler

ruler = nlp.add_pipe("entity_ruler", before="ner")

patterns = [
    {"label": "SKILL", "pattern": [{"LOWER": "react.js"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "react"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "machine learning"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "ml"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "python"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "fastapi"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "nodejs"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "node.js"}]}
]

# 3. Inject the patterns into the ruler
ruler.add_patterns(patterns)

# 4. Test the pipeline on a sample candidate summary
sample_resume_text = "Experienced backend developer specializing in Python and FastAPI. Familiar with Machine Learning and React.js."
doc = nlp(sample_resume_text)

# 5. Extract only our custom 'SKILL' entities
extracted_skills = [ent.text for ent in doc.ents if ent.label_ == "SKILL"]

print("Raw text:", sample_resume_text)
print("Extracted Skills:", extracted_skills)

Raw text: Experienced backend developer specializing in Python and FastAPI. Familiar with Machine Learning and React.js.
Extracted Skills: ['Python', 'FastAPI', 'React.js']


In [7]:
# Cell 5: Segmented Weighted Scoring
"""
#this cell calculates the score for the resume of the candidate based on the weights set by the hiring manager
# the weights are taken from the frontend if assigned and takes into account those weights along with the matching

"""
# 1. Normalization Dictionary
degree_mapping = {
    "b.tech": "bachelor's degree",
    "b.s.": "bachelor's degree",
    "bsc": "bachelor's degree",
    "b.a.": "bachelor's degree",
    "m.tech": "master's degree",
    "m.s.": "master's degree",
    "msc": "master's degree"
}

def normalize_text(text_list, mapping_dict):
    """Replaces specific abbreviations with standard terms to improve semantic matching."""
    normalized_list = []
    for item in text_list:
        # Convert to lowercase for reliable matching
        lower_item = item.lower()
        
        # Check if any abbreviation exists in the string
        for abbreviation, standard_term in mapping_dict.items():
            if abbreviation in lower_item:
                # Replace it (e.g., "b.tech in cs" -> "bachelor's degree in cs")
                lower_item = lower_item.replace(abbreviation, standard_term)
                break # Break after the first match to avoid double-replacements
                
        normalized_list.append(lower_item)
    return normalized_list

def calculate_composite_score(candidate_data, jd_data, weights, thresholds):
    """
    Calculates a weighted composite fit score using section-specific thresholds.
    """
    total_score = 0.0
    section_scores = {}
    
    # Define the sections we want to compare
    sections = ['skills', 'experience', 'education']
    
    for section in sections:
        cand_items = candidate_data.get(section, [])
        jd_items = jd_data.get(section, [])
        
        # Apply normalization ONLY to the education section
        if section == 'education':
            cand_items = normalize_text(cand_items, degree_mapping)
            jd_items = normalize_text(jd_items, degree_mapping)
            
        # Handle empty sections
        if not jd_items:
            continue 
        if not cand_items:
            section_scores[section] = 0.0
            continue 
            
        # Vectorize and Compare
        cand_vecs = embedder.encode(cand_items, convert_to_tensor=True)
        jd_vecs = embedder.encode(jd_items, convert_to_tensor=True)
        sim_matrix = util.cos_sim(jd_vecs, cand_vecs)
        
        # Pull the specific threshold for this section (default to 0.75 as a fallback)
        section_threshold = thresholds.get(section, 0.75)
        
        # Calculate Section Sub-Score
        matches = 0
        for i in range(len(jd_items)):
            best_match_score = sim_matrix[i].max().item()
            if best_match_score >= section_threshold:
                matches += 1
                
        sub_score = (matches / len(jd_items)) * 100
        section_scores[section] = sub_score
        
        # Apply the Hiring Manager's Weight
        weighted_contribution = sub_score * (weights.get(section, 0) / 100.0)
        total_score += weighted_contribution
        
    return round(total_score, 2), section_scores

In [14]:
# Cell 6: End-to-End Beta Test

# 1. Raw Text Inputs (Simulating the email ingestion and UI upload)
raw_jd_text = "Looking for a backend engineer. Must have strong skills in Python, FastAPI, and Machine Learning."

raw_resume_text = "I am a developer with 3 years of experience. I build APIs with FastAPI and NodeJS, and I am currently studying ML."

# 2. Information Extraction (The spaCy step)
print("Extracting entities...")
jd_doc = nlp(raw_jd_text)
resume_doc = nlp(raw_resume_text)

# Extracting the SKILL entities recognized by our custom EntityRuler
jd_skills_extracted = [ent.text for ent in jd_doc.ents if ent.label_ == "SKILL"]
resume_skills_extracted = [ent.text for ent in resume_doc.ents if ent.label_ == "SKILL"]

# 3. Structuring the JSON Payloads
parsed_jd = {
    "skills": jd_skills_extracted,
    "experience": ["3 years backend development"], # Mocked for this test
    "education": ["Computer Science Degree"]       # Mocked for this test
}

parsed_resume = {
    "skills": resume_skills_extracted,
    "experience": ["3 years backend development"], # Mocked for this test
    "education": ["B.Tech in Computer Science"]    # Mocked for this test
}

print(f"JD Skills Found: {parsed_jd['skills']}")
print(f"Resume Skills Found: {parsed_resume['skills']}\n")

# 4. Manager Weights Configuration
test_weights = {
    "skills": 50,       # 50%
    "experience": 30,   # 30%
    "education": 20     # 20%
}

# 5. Run the Scoring Engine 
# (Assuming 0.75 is the "winning" threshold we hypothetically found via evaluate_thresholds)
print("Calculating semantic match...")
final_score, breakdown = calculate_composite_score(
    parsed_resume, 
    parsed_jd, 
    test_weights, 
    {'education':0.75}
)

# 6. Display Results
print("\n--- Beta Test Results ---")
print(f"Composite Fit Score: {final_score}/100")
print(f"Section Breakdown: {breakdown}")

Extracting entities...
JD Skills Found: ['Python', 'FastAPI']
Resume Skills Found: ['FastAPI', 'NodeJS', 'ML']

Calculating semantic match...

--- Beta Test Results ---
Composite Fit Score: 75.0/100
Section Breakdown: {'skills': 50.0, 'experience': 100.0, 'education': 100.0}
